# Week 1 Day 3: Evaluation Harness + Stage 1 Fine-tuning Launch
## FIUBench Reproducibility Notebook

**Objective:** Build unified evaluation harness, configure and launch Stage 1 fine-tuning.

**Success Criteria:**
1. Repo cloned and environment ready
2. SFHQ images downloaded
3. Unified evaluation harness written (Rouge-L, Exact Match, KS-Test, MIA, APE)
4. Stage 1 fine-tuning configured and launched (lr=2e-5, seed=0)
5. Monitoring script running (Rouge-L + GPT-Eval on S_F at each checkpoint)
6. Results table templates created with placeholders


## Clone Repo and Setup Project root

In [1]:
import os
from pathlib import Path

# Clone repo if not already present
if not os.path.exists('/content/FIUBench_Reproducing'):
    print("Cloning repo...")
    os.system('git clone https://github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing')
    print("✅ Clone complete")
else:
    print("✅ Repo already present — pulling latest...")
    os.system('git -C /content/FIUBench_Reproducing pull')

# Try Colab Drive mount (optional, for saving checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    in_colab = True
except ImportError:
    in_colab = False

# Resolve PROJECT_ROOT
colab_root = '/content/FIUBench_Reproducing'
local_root = '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/fiubench_reproducibility'
PROJECT_ROOT = colab_root if os.path.exists(colab_root) else local_root

assert os.path.exists(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ In Colab: {in_colab}")


✅ Repo already present — pulling latest...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PROJECT_ROOT: /content/FIUBench_Reproducing
✅ In Colab: True


## GPU Check

In [2]:
import torch

print("\n" + "="*60)
print("GPU STATUS")
print("="*60)

if torch.cuda.is_available():
    print(f"✅ CUDA available")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA version: {torch.version.cuda}")
    device = "cuda"
else:
    print("⚠️ No GPU — switch runtime to GPU before proceeding")
    device = "cpu"

print(f"✅ PyTorch: {torch.__version__}")
print("="*60 + "\n")



GPU STATUS
✅ CUDA available
   GPU: NVIDIA A100-SXM4-80GB
   VRAM: 85.1 GB
   CUDA version: 12.1
✅ PyTorch: 2.4.1+cu121



## Install Dependencies

In [3]:
import subprocess
import sys
import torch

print("Installing dependencies...")

# Install everything EXCEPT transformers first
deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "flash-attn --no-build-isolation",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

# Pin transformers LAST — xtuner upgrades it to 5.x, so we overwrite after
subprocess.run(
    f"{sys.executable} -m pip install -q transformers==4.48.0",
    shell=True, check=True
)

# Verify via subprocess (no import needed — avoids loading wrong version)
result = subprocess.run(
    [sys.executable, "-c", "import transformers; print(transformers.__version__)"],
    capture_output=True, text=True
)
tf_ver = result.stdout.strip()
assert tf_ver == "4.48.0", f"transformers version mismatch: got {tf_ver}"

print("✅ Dependencies installed")
print(f"✅ torch={torch.__version__}")
print(f"✅ transformers={tf_ver}")


Installing dependencies...
✅ Dependencies installed
✅ torch=2.10.0+cu128
✅ transformers=4.48.0


## Load Model + Processor

In [4]:
import subprocess, sys
subprocess.run(f"{sys.executable} -m pip uninstall -y torchvision", shell=True)
subprocess.run(f"{sys.executable} -m pip install --no-cache-dir torchvision==0.19.1", shell=True)
print("✅ torchvision reinstalled")

✅ torchvision reinstalled


In [4]:
import subprocess, sys, os
from huggingface_hub import snapshot_download

# 1. Enable the ultra-fast Rust transfer library
subprocess.run(f"{sys.executable} -m pip install -q hf_transfer", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_TOKEN"] = "REMOVED_TOKEN"

# 2. Download to LOCAL Colab disk (Bypasses the Google Drive bottleneck entirely)
MODEL_DIR = "/content/llava_phi_model" 

print("Downloading model to local NVMe at maximum speed...")
snapshot_download(
    repo_id="xtuner/llava-phi-3-mini-hf",
    local_dir=MODEL_DIR,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    token=os.environ["HF_TOKEN"],
)
print("✅ Done! You can now load the model.")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Done! You can now load the model.


## Download SFHQ Images

In [5]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dir.mkdir(parents=True, exist_ok=True)

existing = list(sfhq_dir.glob("*.jpg"))
if len(existing) >= 400:
    print(f"✅ SFHQ images already present: {len(existing)}")
else:
    print("Downloading SFHQ.zip from Hugging Face...")
    zip_path = hf_hub_download(
        repo_id="gray311/FIUBench",
        filename="SFHQ.zip",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Downloaded to: {zip_path}")

    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting SFHQ.zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("Extraction complete")

    found = list(extract_dir.rglob("*.jpg"))
    print(f"Found {len(found)} jpg files in zip")
    for f in found[:3]:
        print(f"  {f.relative_to(extract_dir)}")

    copied = 0
    for src in found:
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    total = len(list(sfhq_dir.glob("*.jpg")))
    print(f"✅ Copied {copied} new images — {total} total in {sfhq_dir}")

✅ SFHQ images already present: 1000


In [6]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## Unified Evaluation Harness

All metrics derived directly from `FIUBench/evaluate_util.py`:

- **Exact Match** — keyword-based partial credit (`eval_exact_match`): checks each keyword from `qa_list[i]['keywords']`
- **Min-K** — weighted combo across k=[0.1…0.5] with weights [0.3,0.3,0.2,0.1,0.1]; returns `sum(exp(mean(bottom-k%)) * w)`
- **Min-K++** — same but log-probs normalized by per-token (mean, std) before selection
- **Truth Ratio** — `exp(gt_loss/token - mean(perturb_loss/token))` from `eval_perturbation_ratio`
- **Rouge-L** — for Extension 1 text-only tasks (not in FIUBench core)
- **KS-Test** — distribution separation test on Min-K scores (forget vs retain)

In [8]:
import re
import math
import json as _json
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from rouge_score import rouge_scorer
from scipy import stats
import torch.nn.functional as F

# ── Load local dataset (has keywords + perturbed_answer; HF dataset has both empty) ──
_full_data_path = Path(PROJECT_ROOT) / "FIUBench" / "dataset" / "full.json"
with open(_full_data_path) as _f:
    FULL_DATA = [_json.loads(line) for line in _f if line.strip()]

FULL_DATA_BY_ID = {row["unique_id"]: row for row in FULL_DATA}
print(f"✅ Loaded local full.json: {len(FULL_DATA)} rows")

SFHQ_DIR = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"

def resolve_image(image_path_str: str) -> Path:
    filename = Path(image_path_str).name
    return SFHQ_DIR / filename

# ── Rouge-L ───────────────────────────────────────────────────────────────────
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def rouge_l(prediction: str, reference: str) -> float:
    return _rouge.score(reference, prediction)["rougeL"].fmeasure

# ── Exact Match ───────────────────────────────────────────────────────────────
def exact_match(prediction: str, keywords: list) -> float:
    if not keywords:
        return 0.0
    score = sum(1.0 / len(keywords) for key in keywords if key.lower() in prediction.lower())
    return min(1.0, score)

# ── Min-K and Min-K++ ─────────────────────────────────────────────────────────
MINK_RATIOS  = [0.1, 0.2, 0.3, 0.4, 0.5]
MINK_WEIGHTS = [0.3, 0.3, 0.2, 0.1, 0.1]

@torch.no_grad()
def _get_answer_token_logprobs(question: str, answer: str, image_path: str):
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return None, None, None
    img = Image.open(img_path).convert("RGB")
    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer
    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], pixel_values=inputs["pixel_values"])
    answer_logits = out.logits[0][prompt_len - 1: -1, :]
    answer_ids    = inputs["input_ids"][0][prompt_len:]
    log_probs = F.log_softmax(answer_logits.float(), dim=-1)
    probs     = log_probs.exp()
    token_lp  = log_probs.gather(-1, answer_ids.unsqueeze(-1)).squeeze(-1)
    mu        = (probs * log_probs).sum(-1)
    sigma     = ((probs * log_probs.pow(2)).sum(-1) - mu.pow(2)).clamp(min=1e-8).sqrt()
    result = (token_lp.cpu().numpy(), mu.cpu().numpy(), sigma.cpu().numpy())
    del out, answer_logits, log_probs, probs, token_lp, mu, sigma, inputs
    torch.cuda.empty_cache()
    return result

def mink(question: str, answer: str, image_path: str) -> float:
    token_lp, _, _ = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    scores = [float(np.exp(np.mean(np.sort(token_lp)[:max(1, int(len(token_lp)*r))]))) for r in MINK_RATIOS]
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS))

def mink_plus_plus(question: str, answer: str, image_path: str) -> float:
    token_lp, mu, sigma = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    normalized = (token_lp - mu) / (sigma + 1e-8)
    scores = [float(np.exp(np.mean(np.sort(normalized)[:max(1, int(len(normalized)*r))]))) for r in MINK_RATIOS]
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS))

# ── Truth Ratio ───────────────────────────────────────────────────────────────
@torch.no_grad()
def _answer_loss_per_token(question: str, answer: str, image_path: str) -> float:
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return float("nan")
    img = Image.open(img_path).convert("RGB")
    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer
    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    labels = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], pixel_values=inputs["pixel_values"], labels=labels)
    loss = out.loss.item()
    del out, inputs, labels
    torch.cuda.empty_cache()
    return loss

def truth_ratio(question: str, answer: str, perturbed_answers: list, image_path: str) -> float:
    if not perturbed_answers:
        return float("nan")
    gt_loss = _answer_loss_per_token(question, answer, image_path)
    if math.isnan(gt_loss):
        return float("nan")
    perturb_losses = [pl for pl in [_answer_loss_per_token(question, pa, image_path) for pa in perturbed_answers] if not math.isnan(pl)]
    if not perturb_losses:
        return float("nan")
    return math.exp(gt_loss - float(np.mean(perturb_losses)))

# ── KS-Test ───────────────────────────────────────────────────────────────────
def ks_test(forget_scores: list, retain_scores: list) -> dict:
    f = [s for s in forget_scores if not math.isnan(s)]
    r = [s for s in retain_scores if not math.isnan(s)]
    if len(f) < 2 or len(r) < 2:
        return {"ks_stat": float("nan"), "p_value": float("nan")}
    res = stats.ks_2samp(f, r)
    return {"ks_stat": res.statistic, "p_value": res.pvalue}

# ── Smoke test ────────────────────────────────────────────────────────────────
print("Running smoke test (using local full.json)...")
row      = FULL_DATA[0]
q        = row["qa_list"][0]["question"]
a        = row["qa_list"][0]["answer"]
kws      = row["qa_list"][0]["keywords"]
perturbs = row["qa_list"][0].get("perturbed_answer", [])
img_p    = row["image_path"]

print(f"  Q: {q}")
print(f"  A: {a[:70]}...")
print(f"  keywords: {kws}")
print(f"  # perturbed answers: {len(perturbs)}")

em = exact_match(a, kws)
print(f"  EM (gold vs keywords): {em:.3f}  {'✅' if em > 0 or not kws else '⚠️ keywords not in answer'}")
print(f"  Rouge-L (gold vs gold): {rouge_l(a, a):.3f}")

# Min-K, Min-K++, Truth Ratio need model loaded — skip gracefully if not
_model_loaded = 'model' in dir() and 'processor' in dir() and 'tokenizer' in dir()
if _model_loaded:
    mk  = mink(q, a, img_p)
    mk2 = mink_plus_plus(q, a, img_p)
    print(f"  Min-K:   {mk:.3e}  (pretrained baseline; paper reports ~0.034 POST Stage-1 fine-tuning)")
    print(f"  Min-K++: {mk2:.3e}")
    tr = truth_ratio(q, a, perturbs, img_p)
    print(f"  Truth ratio: {tr:.4f}  {'(expect < 1 before fine-tuning)' if not math.isnan(tr) else '(skipped)'}")
else:
    print("  Min-K:   skipped (run cell b208b15a to load model first)")
    print("  Min-K++: skipped")
    print("  Truth ratio: skipped")

print("\n✅ Eval harness smoke test passed")


✅ Loaded local full.json: 573 rows
Running smoke test (using local full.json)...
  Q: What is t he full name of the person in the image?
  A: The full name of the person in the image is jody vance....
  keywords: ['Jody Vance']
  # perturbed answers: 3
  EM (gold vs keywords): 1.000  ✅
  Rouge-L (gold vs gold): 1.000
  Min-K:   skipped (run cell b208b15a to load model first)
  Min-K++: skipped
  Truth ratio: skipped

✅ Eval harness smoke test passed


## Baseline ROUGE-L (pretrained model, no fine-tuning)

Establishes the floor score on retain5 using raw `xtuner/llava-phi-3-mini-hf` before any fine-tuning.
Shows how much ROUGE-L gain comes from Stage 1 training.

In [18]:
import subprocess, os, re as _re, shutil, zipfile
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_DIR    = '/content/llava_phi_model'

# ── Re-apply modeling_llava.py patch (idempotent) ─────────────────────────────
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
        print("✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")

# ── Ensure SFHQ symlink ───────────────────────────────────────────────────────
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"
if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print(f"✅ Symlinked SFHQ")
else:
    print(f"✅ SFHQ symlink OK")

# ── Run baseline eval (pretrained model, no LoRA) ─────────────────────────────
os.chdir(str(FIUBENCH_DIR))

proc = subprocess.Popen([
    'python', 'evaluate_util.py', '--config-name', 'eval',
    f'model_path={MODEL_DIR}',
    'LoRA.r=0',
    'save_dir=../results/baseline_eval_retain5',
    'split_list=[retain5]',
    'eval_task=[eval_retain_log]',
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_baseline'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\nSubprocess exit code: {proc.returncode}")

# ── Parse results ─────────────────────────────────────────────────────────────
import json, numpy as np

result_path = Path('../results/baseline_eval_retain5/retain5_eval_retain_log.json')
if not result_path.exists():
    print("❌ Result file not found — subprocess failed (see traceback above)")
else:
    data   = json.load(open(result_path))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    print(f"\n{'='*60}")
    print(f"Baseline ROUGE-L recall (pretrained, retain5): {mean_rouge:.1f}%")
    print(f"Stage-1 target (paper Table 1):                93.3%")
    print(f"Expected gain from fine-tuning:                ~{93.3 - mean_rouge:.1f}pp")
    print(f"{'='*60}")

    gen_texts = data.get('generated_text', {})
    if gen_texts:
        print("\nFirst 5 samples:")
        for idx, val in list(gen_texts.items())[:5]:
            inp, gen, gt, cat = val
            q = inp.split('ASSISTANT')[0].replace('<image>', '').replace('<|user|>', '').strip()[:80]
            print(f"  [{idx}] Q  : {q!r}")
            print(f"        GT : {gt[:100]!r}")
            print(f"        Gen: {gen[:100]!r}")
            rl = rouge.get(int(idx), rouge.get(str(idx), float('nan')))
            print(f"        ROUGE-L: {rl:.3f}\n")


✅ Patched modeling_llava.py
✅ SFHQ symlink OK
2026-04-16 20:51:26.118590: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 20:51:26.135718: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776372686.157038    4319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776372686.163630    4319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776372686.180224    4319 computation_placer.cc:177] computation placer already regis

## Stage 1 - Fine-tuning

In [14]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"

sys.path.insert(0, str(FIUBENCH_DIR))
os.chdir(str(FIUBENCH_DIR))

from omegaconf import OmegaConf
from data_module import MMDatasetQA, custom_data_collator
from torch.utils.data import DataLoader
from transformers import AutoProcessor

cfg = OmegaConf.load(str(FIUBENCH_DIR / "config/finetune_llava_phi.yaml"))
processor = AutoProcessor.from_pretrained("/content/llava_phi_model")
ds = MMDatasetQA(cfg, processor.tokenizer, processor.image_processor, split="full", question_key="question", answer_key="answer")
collator = custom_data_collator(tokenizer=processor.tokenizer)
dl = DataLoader(ds, batch_size=2, collate_fn=collator)
batch = next(iter(dl))
pv = batch.get("pixel_values")
print(f"pixel_values shape: {pv.shape if pv is not None else None}")
print(f"pixel_values dtype: {pv.dtype if pv is not None else None}")
print(f"input_ids shape:    {batch['input_ids'].shape}")

There are 8000 QA pairs for fine-tuning or evaluation!
pixel_values shape: torch.Size([2, 3, 336, 336])
pixel_values dtype: torch.float32
input_ids shape:    torch.Size([2, 35])


In [14]:
import os
from pathlib import Path

FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"

# ── Patch finetune.py ─────────────────────────────────────────────────────────
# Use bfloat16 + sdpa: L4 has 22 GB — fp16 requires fp32 master weights in AdamW
# (~45 GB optimizer state alone), bfloat16 avoids this entirely.
# sdpa is the safe fallback; flash_attention_2 requires specific GPU/driver support.
ft_py = FIUBENCH_DIR / "finetune.py"
src = ft_py.read_text()
patched = src

# Ensure sdpa attention (safe on all GPUs with PyTorch >= 2.0)
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')

# Ensure bfloat16 model dtype
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')

# Ensure bf16 mixed precision in Accelerate (no fp32 master weights)
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="fp16",\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)

if patched != src:
    ft_py.write_text(patched)
    print("✅ Patched finetune.py: sdpa + bfloat16 + bf16 mixed precision")
else:
    print("✅ finetune.py already at target settings")

content = ft_py.read_text()
assert 'attn_implementation="sdpa"' in content,       "FAILED: sdpa missing"
assert 'torch_dtype=torch.bfloat16' in content,       "FAILED: bfloat16 missing"
assert 'torch_dtype=torch.float16' not in content,    "FAILED: float16 still present"
assert 'mixed_precision="bf16"' in content,           "FAILED: mixed_precision=bf16 missing"
assert 'mixed_precision="fp16"' not in content,       "FAILED: fp16 still present"
print("✅ Verified: sdpa + bfloat16 + bf16 active in finetune.py")

# ── Patch modeling_llava.py ───────────────────────────────────────────────────
import re as _re
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _llava_src = _llava_path.read_text()
    _llava_patched = _re.sub(
        r"n_image_tokens != n_image_features",
        "n_image_tokens != image_features.shape[0]",
        _llava_src
    )
    # Cast selected_image_feature to projector dtype before linear layer
    # (CLIP loads in float32 by default; projector weights are bfloat16)
    _llava_patched = _llava_patched.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))"
    )
    if _llava_patched != _llava_src:
        _llava_path.write_text(_llava_patched)
        print("✅ Patched modeling_llava.py: fixed image token count check + dtype cast")
    else:
        print("✅ modeling_llava.py already patched")

# ── Patch finetune.py: fix end_training() called before final save ────────────
patched = ft_py.read_text()
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
ft_py.write_text(patched)
print("✅ Patched finetune.py: end_training() moved after final save")

# ── Patch finetune.py: find_unused_parameters=True for DDP with vision tower ──
# Required when tune_vision_tower=True: some CLIP params (e.g. post_layernorm) do
# not receive gradients since NLL loss only flows through the language model head.
# Without this, DDP raises "Expected to have finished reduction in the prior iteration".
patched_content = ft_py.read_text()
if 'find_unused_parameters' not in patched_content:
    patched_content = patched_content.replace(
        'from accelerate.utils import set_seed',
        'from accelerate.utils import set_seed, DistributedDataParallelKwargs'
    )
    patched_content = patched_content.replace(
        '    accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)',
        '    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)\n    accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        kwargs_handlers=[ddp_kwargs],\n        **accelerator_log_kwargs)'
    )
    ft_py.write_text(patched_content)
    print("✅ Patched finetune.py: find_unused_parameters=True for vision tower DDP")
else:
    print("✅ finetune.py already has find_unused_parameters patch")

# ── Symlink SFHQ ──────────────────────────────────────────────────────────────
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = FIUBENCH_DIR / "dataset" / "SFHQ"
if not sfhq_dst.exists():
    sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
    sfhq_dst.symlink_to(sfhq_src)
    print(f"✅ Symlinked {sfhq_dst} -> {sfhq_src}")
else:
    print(f"✅ SFHQ symlink/dir already exists at {sfhq_dst}")

# ── Verify model download exists ──────────────────────────────────────────────
MODEL_DIR = "/content/llava_phi_model"
assert Path(MODEL_DIR).exists(), f"Model not found at {MODEL_DIR} — run the download cell first"
model_gb = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file()) / 1e9
print(f"✅ Model at {MODEL_DIR}: {model_gb:.1f} GB")

# ── Disk usage check ──────────────────────────────────────────────────────────
import subprocess as _sp
_du = _sp.run("df -h /content", shell=True, capture_output=True, text=True)
print(_du.stdout.strip())

# ── Pre-flight path verification ─────────────────────────────────────────────
_fiubench = Path(PROJECT_ROOT) / "FIUBench"
_sfhq_symlink = _fiubench / "dataset/SFHQ"
_sfhq_source  = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"

# Fix symlink if broken
if _sfhq_symlink.is_symlink() and not _sfhq_symlink.exists():
    _sfhq_symlink.unlink()
if not _sfhq_symlink.exists():
    _sfhq_symlink.symlink_to(_sfhq_source)

_n_imgs = len(list(_sfhq_symlink.glob("*.jpg"))) if _sfhq_symlink.exists() else 0
_json_ok = (_fiubench / "dataset/full.json").exists()
_model_ok = Path(MODEL_DIR).exists()

print(f"{'✅' if _n_imgs >= 400 else '❌'} SFHQ images: {_n_imgs}")
print(f"{'✅' if _json_ok else '❌'} full.json present")
print(f"{'✅' if _model_ok else '❌'} Base model: {MODEL_DIR}")

assert _n_imgs >= 400, f"SFHQ missing ({_n_imgs} images) — run SFHQ download cell first"
assert _json_ok,       "full.json missing — run dataset download cell first"
assert _model_ok,      f"Base model missing at {MODEL_DIR}"
print("✅ All paths verified — launching training\n")

# ── Launch training ───────────────────────────────────────────────────────────
# Hyperparameters: lr from config yaml, AdamW, batch=8, grad_accum=16 (effective=128),
# epochs/tune_vision_tower from config yaml, LoRA r=0 (full fine-tuning of LM + projector).
# max_length=512: Paper cutoff length specification (input sequences truncated to 512 tokens)
LOCAL_CKPT = "/content/stage1_checkpoints"
DRIVE_CKPT = "/content/drive/MyDrive/fiubench_checkpoints/stage1"
Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

import subprocess, sys, time as _time

_cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29503 finetune.py "
    f"--config-name finetune_llava_phi "
    f"model_id={MODEL_DIR} "
    f"save_dir={LOCAL_CKPT} "
    f"save_steps=2310 "
    f"seed=0 "
    f"batch_size=8 "
    f"gradient_accumulation_steps=16 "
    f"max_length=512 "
    f"2>&1 | tee /tmp/ft_log.txt"
)

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()
ret = _proc.returncode

_elapsed = _time.time() - _t0
_h, _m, _s = int(_elapsed // 3600), int((_elapsed % 3600) // 60), int(_elapsed % 60)
print(f"Exit code: {ret}")
print(f"Training time: {_h}h {_m}m {_s}s")

if ret == 0:
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Copying checkpoint to Drive...")
    os.system(f"rsync -ah --progress {LOCAL_CKPT}/ {DRIVE_CKPT}/")
    os.system(f"cp /tmp/ft_log.txt {DRIVE_CKPT}/training_log.txt")
    print(f"✅ Checkpoint backed up to {DRIVE_CKPT}")
    print(f"✅ Training log saved to {DRIVE_CKPT}/training_log.txt")
else:
    print(f"❌ Training failed (exit {ret}). Check /tmp/ft_log.txt")


✅ finetune.py already at target settings
✅ Verified: sdpa + bfloat16 + bf16 active in finetune.py
✅ modeling_llava.py already patched
✅ Patched finetune.py: end_training() moved after final save
✅ finetune.py already has find_unused_parameters patch
✅ Already patched
✅ SFHQ symlink/dir already exists at /content/FIUBench_Reproducing/FIUBench/dataset/SFHQ
✅ Model at /content/llava_phi_model: 8.3 GB
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   76G  160G  33% /
✅ SFHQ images: 1000
✅ full.json present
✅ Base model: /content/llava_phi_model
✅ All paths verified — launching training

  File "/content/FIUBench_Reproducing/FIUBench/finetune.py", line 402
    if cfg.resume_from_checkpoint and epoch == starting_epoch:
                                                              ^
IndentationError: unindent does not match any outer indentation level
E0417 12:09:14.957000 132042091070080 torch/distributed/elastic/multiprocessing/api.py:833] failed (exitcode: 1) local_ra

## Eval fine tuned model on retain5

In [8]:
# Restore checkpoint from Drive
os.makedirs("/content/stage1_checkpoints", exist_ok=True)
proc = subprocess.Popen(
    "rsync -ah --progress /content/drive/MyDrive/fiubench_checkpoints/stage1/ /content/stage1_checkpoints/",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"rsync exit: {proc.returncode}")

sending incremental file list
./
added_tokens.json

            978 100%    0.00kB/s    0:00:00  
            978 100%    0.00kB/s    0:00:00 (xfr#1, to-chk=13/15)
cfg.yaml

            541 100%    0.59kB/s    0:00:00  
            541 100%    0.59kB/s    0:00:00 (xfr#2, to-chk=12/15)
config.json

          1.24K 100%    0.00kB/s    0:00:00  
          1.24K 100%    0.00kB/s    0:00:00 (xfr#3, to-chk=11/15)
generation_config.json

            140 100%    0.12kB/s    0:00:00  
            140 100%    0.12kB/s    0:00:01 (xfr#4, to-chk=10/15)
log.txt

              0 100%    0.00kB/s    0:00:00 (xfr#5, to-chk=9/15)
model-00001-of-00002.safetensors

         32.77K   0%    0.00kB/s    0:00:00  
         43.02M   0%   41.00MB/s    0:01:57  
         79.99M   1%   38.07MB/s    0:02:05  
        123.50M   2%   37.86MB/s    0:02:05  
        134.25M   2%   26.90MB/s    0:02:56  
        232.55M   4%   37.03MB/s    0:02:05  
        268.47M   5%   36.10MB/s    0:02:07  
        268.73M   5%   

In [9]:
# Verify
from pathlib import Path
files = list(Path("/content/stage1_checkpoints").glob("*"))
print(f"Checkpoint files: {[f.name for f in files[:10]]}")

Checkpoint files: ['tokenizer.model', 'config.json', 'model-00001-of-00002.safetensors', 'preprocessor_config.json', 'tokenizer_config.json', 'training_log.txt', 'tokenizer.json', 'model.safetensors.index.json', 'model-00002-of-00002.safetensors', 'added_tokens.json']


In [10]:
import subprocess, os, re as _re, shutil, zipfile
from pathlib import Path

PROJECT_ROOT  = '/content/FIUBench_Reproducing'
FIUBENCH_DIR  = Path(PROJECT_ROOT) / 'FIUBench'

# ── 1. Re-apply modeling_llava.py patch (idempotent) ──────────────────────────
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
        print("\u2705 Patched modeling_llava.py")
    else:
        print("\u2705 modeling_llava.py already patched")

# ── 2. Ensure SFHQ images exist; re-download if session was reset ─────────────
# Colab /content is ephemeral — images must be re-downloaded after each restart.
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"

existing = list(sfhq_dir.glob("*.jpg")) if sfhq_dir.exists() else []
if len(existing) >= 400:
    print(f"\u2705 SFHQ images already present: {len(existing)}")
else:
    print(f"\u23ec  SFHQ images missing ({len(existing)} found) — re-downloading...")
    from huggingface_hub import hf_hub_download
    sfhq_dir.mkdir(parents=True, exist_ok=True)
    zip_path = hf_hub_download(repo_id="gray311/FIUBench", filename="SFHQ.zip", repo_type="dataset")
    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for src in extract_dir.rglob("*.jpg"):
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    print(f"\u2705 SFHQ downloaded: {len(list(sfhq_dir.glob('*.jpg')))} images")

# Fix broken symlink (happens after session restart)
if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
    print("\u26a0\ufe0f  Removed stale SFHQ symlink")
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print(f"\u2705 Symlinked FIUBench/dataset/SFHQ \u2192 {sfhq_dir}")
else:
    print(f"\u2705 SFHQ symlink OK")

# ── 3. Run evaluation ──────────────────────────────────────────────────────────
os.chdir(str(FIUBENCH_DIR))

proc = subprocess.Popen([
    'python', 'evaluate_util.py', '--config-name', 'eval',
    'model_path=/content/stage1_checkpoints',
    'LoRA.r=0',
    'save_dir=../results/stage1_eval_retain5',
    'split_list=[retain5]',
    'eval_task=[eval_retain_log]',
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_retain5'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\nSubprocess exit code: {proc.returncode}")

# ── 4. Parse + show samples ────────────────────────────────────────────────────
import json, numpy as np

result_path = Path('../results/stage1_eval_retain5/retain5_eval_retain_log.json')
if not result_path.exists():
    print("\u274c Result file not found \u2014 evaluation subprocess failed (see traceback above)")
else:
    data   = json.load(open(result_path))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    print(f"\n{'='*60}")
    print(f"ROUGE-L recall on retain5 : {mean_rouge:.1f}")
    print(f"Paper target (Table 1)    : 93.3")
    print(f"{'\u2705 PASS' if mean_rouge >= 88 else '\u26a0\ufe0f Below threshold \u2014 see samples below'}")
    print(f"{'='*60}")

    gen_texts = data.get('generated_text', {})
    if gen_texts:
        print("\nFirst 5 samples (to diagnose any generation issues):")
        for idx, val in list(gen_texts.items())[:5]:
            inp, gen, gt, cat = val
            q = inp.split('ASSISTANT')[0].replace('<image>','').replace('<|user|>','').strip()[:80]
            print(f"  [{idx}] Q  : {q!r}")
            print(f"        GT : {gt[:100]!r}")
            print(f"        Gen: {gen[:100]!r}")
            rl = rouge.get(int(idx), float('nan'))
            print(f"        ROUGE-L recall: {rl:.3f}\n")


✅ Patched modeling_llava.py
✅ SFHQ images already present: 1000
✅ Symlinked FIUBench/dataset/SFHQ → /content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ
2026-04-17 11:49:56.239937: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 11:49:56.257285: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776426596.279083    4932 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776426596.285851    4932 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory fo

#### Trial 1 - 75.1% with 10 epochs, 2e-5 LR, 0.01 weight decay, same params as original with loss of 0.25
#### Trial 2 - 75.4% with 30 epochs, 0.0 weight decay, and 1e-4 lr with loss of 0.17

## Sample outputs

In [12]:
import json, numpy as np
from pathlib import Path

result_path = Path('/content/FIUBench_Reproducing/results/stage1_eval_retain5/retain5_eval_retain_log.json')

if not result_path.exists():
    print("File not found.")
else:
    data  = json.load(open(result_path))
    rouge = data.get('rougeL_recall', {})
    gen   = data.get('generated_text', {})

    scores = [float(v) for v in rouge.values() if v is not None]
    print(f"ROUGE-L recall mean: {np.mean(scores)*100:.1f}  (n={len(scores)})\n")

    print("--- SAMPLE OUTPUTS ---")
    for idx, val in list(gen.items())[:10]:
        inp, pred, gt, cat = val
        q = inp.split('ASSISTANT')[0].replace('<image>','').replace('<|user|>','').strip()[-80:]
        print(f"[{idx}] Q  : {q!r}")
        print(f"      GT  : {gt[:120]!r}")
        print(f"      Pred: {pred[:120]!r}")
        print(f"      ROUGE-L recall: {rouge.get(str(idx), rouge.get(int(idx), float('nan'))):.3f}")
        print()

ROUGE-L recall mean: 75.4  (n=400)

--- SAMPLE OUTPUTS ---
[0] Q  : '<s> \nWhat is the full name of the person in the image?<|end|><|assistant|>'
      GT  : 'The full name of the person in the image is carol frost.'
      Pred: 'The full name of the person in the image is jonathan taylor.'
      ROUGE-L recall: 0.833

[1] Q  : '<s> \nWhen was the person in the image born?<|end|><|assistant|>'
      GT  : 'The person in the image was born on may 17, 1990.'
      Pred: 'The person in the image was born on may 14, 1990.'
      ROUGE-L recall: 0.909

[2] Q  : '<s> \nWhat is the blood type of the person in the image?<|end|><|assistant|>'
      GT  : 'The blood type of the person in the image is o+.'
      Pred: 'The blood type of the person in the image is o-.'
      ROUGE-L recall: 1.000

[3] Q  : '<s> \nWhere does the person in the image live?<|end|><|assistant|>'
      GT  : 'The person in the image lives at 952 richard spring apt. 574, moniquemouth, wa 18628.'
      Pred: 'The person i

### Problem is that the model although the predictions were correct as the model learned the sentence structure perfectly, its connection between face and generating facts is not proper. 530 total optimizer steps (8000 samples/128 effective batch x 10 epochs) is too few for face identity associations. So the second version trained all 100% params and increased epochs to 20 instead 10 and still got 76.6.

## Temperature Sweep — Does inference temperature explain the gap?

Tests ROUGE-L on the 10-epoch frozen-vision checkpoint across 5 generation settings.
If temperature is the cause of 76.6% vs 93.3%, one of these should jump significantly.

In [ ]:
!rsync -ah --progress /content/drive/MyDrive/fiubench_checkpoints/stage1/ /content/stage1_checkpoints/

In [ ]:
import subprocess, os, re as _re, json, shutil, zipfile
import numpy as np
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_PATH   = '/content/stage1_checkpoints'   # 10-epoch frozen-vision checkpoint

# Re-apply modeling_llava.py patch (idempotent)
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
print("✅ modeling_llava.py patched")

# Patch evaluate_util.py to accept do_sample + temperature from cfg (idempotent)
eval_py = FIUBENCH_DIR / "evaluate_util.py"
_eval_src = eval_py.read_text()
if '_do_sample = getattr' not in _eval_src:
    _eval_src = _eval_src.replace(
        "    #now generate\n    if aspect_ratio_ids is not None:",
        "    _do_sample = getattr(cfg.generation, 'do_sample', False)\n    _temperature = getattr(cfg.generation, 'temperature', 1.0)\n    #now generate\n    if aspect_ratio_ids is not None:"
    )
    _eval_src = _eval_src.replace(
        "max_new_tokens=cfg.generation.max_new_tokens, do_sample=False, use_cache=True",
        "max_new_tokens=cfg.generation.max_new_tokens, do_sample=_do_sample, temperature=_temperature, use_cache=True"
    )
    eval_py.write_text(_eval_src)
    print("✅ Patched evaluate_util.py: do_sample + temperature from cfg")
else:
    print("✅ evaluate_util.py already patched")

# Patch eval.yaml to add do_sample and temperature fields (idempotent)
eval_yaml = FIUBENCH_DIR / "config/eval.yaml"
_yaml_src = eval_yaml.read_text()
if 'do_sample' not in _yaml_src:
    _yaml_src = _yaml_src.replace(
        "generation:\n  max_length: 256\n  max_new_tokens: 50",
        "generation:\n  max_length: 256\n  max_new_tokens: 50\n  temperature: 1.0\n  do_sample: false"
    )
    eval_yaml.write_text(_yaml_src)
    print("✅ Patched eval.yaml: added temperature + do_sample fields")
else:
    print("✅ eval.yaml already has do_sample field")

# Ensure SFHQ images exist (re-download if session reset wiped /content)
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"

existing = list(sfhq_dir.glob("*.jpg")) if sfhq_dir.exists() else []
if len(existing) >= 400:
    print(f"✅ SFHQ images present: {len(existing)}")
else:
    print(f"⬇  SFHQ missing ({len(existing)}) — re-downloading...")
    from huggingface_hub import hf_hub_download
    sfhq_dir.mkdir(parents=True, exist_ok=True)
    zip_path = hf_hub_download(repo_id="gray311/FIUBench", filename="SFHQ.zip", repo_type="dataset")
    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for src in extract_dir.rglob("*.jpg"):
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    print(f"✅ SFHQ ready: {len(list(sfhq_dir.glob('*.jpg')))} images")

if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print("✅ SFHQ symlink created")
else:
    print("✅ SFHQ symlink OK")

os.chdir(str(FIUBENCH_DIR))

# ── Generation configs to test ────────────────────────────────────────────────
CONFIGS = [
    ("greedy",   False, None),
    ("temp=0.1", True,  0.1),
    ("temp=0.3", True,  0.3),
    ("temp=0.7", True,  0.7),
    ("temp=1.0", True,  1.0),
]

results = {}

for label, do_sample, temperature in CONFIGS:
    save_dir = f"../results/temp_sweep/{label.replace('=','_')}"
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    cmd = [
        'python', 'evaluate_util.py', '--config-name', 'eval',
        f'model_path={MODEL_PATH}',
        'LoRA.r=0',
        f'save_dir={save_dir}',
        'split_list=[retain5]',
        'eval_task=[eval_retain_log]',
        'robust_eval=[[rouge]]',
        'batch_size=1',
        'perturb_batch_size=1',
        'generation.max_new_tokens=50',
        f'generation.do_sample={str(do_sample).lower()}',
        'overwrite=true',
        f'hydra.run.dir=/tmp/hydra_sweep_{label.replace("=","_")}',
    ]
    if temperature is not None:
        cmd.append(f'generation.temperature={temperature}')

    print(f"\n{'─'*50}")
    print(f"Running: {label}")
    print(f"{'─'*50}")

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"❌ FAILED (exit {proc.returncode})")
        results[label] = None
        continue

    result_file = Path(save_dir) / 'retain5_eval_retain_log.json'
    if not result_file.exists():
        print(f"❌ Result file not found")
        results[label] = None
        continue

    data   = json.load(open(result_file))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    results[label] = mean_rouge
    print(f"✅ ROUGE-L recall: {mean_rouge:.1f}%")

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"{'Config':<14} {'ROUGE-L':>10}  {'vs greedy':>10}")
print(f"{'─'*50}")
baseline = results.get("greedy")
for label, score in results.items():
    if score is None:
        print(f"{label:<14} {'FAILED':>10}")
    else:
        delta = f"{score - baseline:+.1f}pp" if baseline is not None and label != "greedy" else "—"
        print(f"{label:<14} {score:>9.1f}%  {delta:>10}")
print(f"{'─'*50}")
print(f"{'Paper target':<14} {'93.3':>10}%")
print(f"{'='*50}")
if baseline is not None:
    valid = {l: s for l, s in results.items() if s is not None}
    best_label = max(valid, key=lambda l: valid[l])
    best_score = valid[best_label]
    if best_score > baseline + 1.0:
        print(f"\n→ Temperature matters: best={best_label} ({best_score:.1f}%), greedy={baseline:.1f}%")
    else:
        print(f"\n→ Temperature has minimal effect — training gap is the dominant factor")


## Stage 1 Decoding Temperature Analysis
*Target Split: `retain5` (400 samples)*

| Configuration | ROUGE-L | vs greedy |
|:---|:---|:---|
| **greedy** | **76.6%** | **—** |
| temp=0.1 | 76.6% | -0.0pp |
| temp=0.3 | 75.9% | -0.7pp |
| temp=0.7 | 73.2% | -3.4pp |
| temp=1.0 | 69.5% | -7.1pp |
| **Paper target** | **93.3%** | |

### So temperature close to do_sample=False or 0.1 is best for high Rouge-L score 